In [0]:
import requests
import json
import os
from typing import Dict, List, Optional

def get_reference_data(
    base_url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,
    reference_type: str,
    resource: str,
    resource_code: Optional[str] = None,
    query_params: Optional[Dict[str, str]] = None
):
    """
    Generic function to fetch reference data from the API.
    
    Args:
        base_url: Base API URL
        headers: Authentication headers
        catalog_name: Unity Catalog name
        schema_name: Schema name
        volume_name: Volume name
        reference_type: e.g., 'mds-reference', 'Offers'
        resource: e.g., 'airports', 'countries', 'aircrafts'
        resource_code: Optional specific code (e.g., airport code)
        query_params: Optional query parameters (e.g., {'lang': 'en'})
    """
    
    # Build endpoint
    endpoint_parts = ["/v1", reference_type, resource]
    if resource_code:
        endpoint_parts.append(resource_code)
    
    endpoint = "/".join(endpoint_parts)
    
    # Add query parameters if provided
    if query_params:
        query_string = "&".join([f"{k}={v}" for k, v in query_params.items()])
        endpoint = f"{endpoint}?{query_string}"
    
    # Build directory structure
    dir_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{reference_type}/{resource}"
    if resource_code:
        dir_path = f"{dir_path}/{resource_code}"
    
    # Create filename
    if resource_code:
        filename = f"{dir_path}/{resource_code}.json"
    else:
        filename = f"{dir_path}/all_{resource}.json"
    
    try:
        full_url = base_url + endpoint
        print(f"Fetching: {full_url}")
        
        response = requests.get(full_url, headers=headers)
        
        if response.status_code == 200:
            json_data = response.json()
            
            # Save to file
            with open(filename, "w") as file:
                json.dump(json_data, file, indent=2)
            
            print(f"✓ Saved to: {filename}")
            return json_data
        else:
            print(f"✗ Request failed with status code {response.status_code}")
            print(f"  Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"✗ An error occurred: {e}")
        return None


def fetch_all_reference_data(
    base_url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str
):
    """
    Loop through all reference types and resources to fetch data.
    """
    
    # Define your data structure
    reference_structure = {
        "mds-reference": {
            "countries": [],  # Fetch all countries
            "cities": [],     # Fetch all cities
            "airports": [],   # Fetch all airports
            "aircrafts": []   # Fetch all aircrafts
        },
        "offers": {
            "seatmaps": {
                "requires": ["flightNumber", "origin", "destination", "departureDate", "cabinTypeCode"]
            },
            "lounges": {
                "requires": ["code", "cabinClassCode", "tierCode", "languageCode"]
            }
        }
    }
    
    results = {}
    
    # Fetch MDS Reference Data (these are general lookups)
    for resource in reference_structure["mds-reference"]:
        print(f"\n{'='*60}")
        print(f"Fetching {resource}...")
        print(f"{'='*60}")
        
        result = get_reference_data(
            base_url=base_url,
            headers=headers,
            catalog_name=catalog_name,
            schema_name=schema_name,
            volume_name=volume_name,
            reference_type="mds-reference",
            resource=resource
        )
        
        if result:
            results[resource] = result
    
    return results


# Example usage for specific resources with codes
def fetch_specific_resource(
    base_url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,
    resource_type: str,
    codes: List[str]
):
    """
    Fetch specific resources by their codes.
    
    Example:
        airports = ['FRA', 'LIS', 'DUB']
        fetch_specific_resource(..., 'airports', airports)
    """
    results = {}
    
    for code in codes:
        print(f"\nFetching {resource_type}: {code}")
        result = get_reference_data(
            base_url=base_url,
            headers=headers,
            catalog_name=catalog_name,
            schema_name=schema_name,
            volume_name=volume_name,
            reference_type="mds-reference",
            resource=resource_type,
            resource_code=code
        )
        
        if result:
            results[code] = result
    
    return results



In [0]:
if __name__ == "__main__":
    # Configuration
    password = dbutils.secrets.get(scope="lh-api", key="password")
    
    base_url = "https://lh-proxy.onrender.com"
    headers ={"password": "{password}"}
    
    catalog_name = "data_catalog"
    schema_name = "bronze"
    volume_name = "bronze_volume"
    
    
    # Option 1: Fetch all reference data
    print("Starting data fetch...")
    all_data = fetch_all_reference_data(
        base_url=base_url,
        headers=headers,
        catalog_name=catalog_name,
        schema_name=schema_name,
        volume_name=volume_name
    )
    
    # Option 2: Fetch specific airports
    specific_airports = ['FRA', 'LIS', 'DUB', 'JFK']
    airport_data = fetch_specific_resource(
        base_url=base_url,
        headers=headers,
        catalog_name=catalog_name,
        schema_name=schema_name,
        volume_name=volume_name,
        resource_type="airports",
        codes=specific_airports
    )
    
    # Option 3: Fetch specific aircraft
    specific_aircraft = ['333', '747', 'A380']
    aircraft_data = fetch_specific_resource(
        base_url=base_url,
        headers=headers,
        catalog_name=catalog_name,
        schema_name=schema_name,
        volume_name=volume_name,
        resource_type="aircrafts",
        codes=specific_aircraft
    )
    
    print("\n" + "="*60)
    print("Data fetch complete!")
    print("="*60)